# Exploratory Data Analysis: Raw EPA Data



In [ ]:
# Local project setup
import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")




## Run the stage


Enhanced exploratory data analysis for the raw EPA annual air-quality summary CSV.

Covers:
- Dataset structure and variable types
- Missing-value analysis (counts, percentages, patterns, visualisation)
- Duplicate detection
- Outlier analysis (IQR method + boxplots)
- Class imbalance (for proposed classification target)
- Distributions by pollutant, state, and year
- Correlation analysis (numeric features)
- Target-variable proposal and analysis
- Data-leakage risk identification

Outputs:
- reports/raw_eda/*.csv
- reports/raw_eda/*.png
- reports/raw_eda/raw_eda_report.md



In [ ]:


from __future__ import annotations

import json
import warnings
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)


# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

project_root = Path.cwd().parent
RAW_PATH = project_root / "data" / "raw" / "air_quality.csv"
OUTPUT_DIR = project_root / "reports" / "raw_eda"
CHUNK_SIZE = 100_000
SAMPLE_PER_PARAMETER = 5_000      # stratified sample size per pollutant
RANDOM_STATE = 42

NUMERIC_COLUMNS = [
    "latitude",
    "longitude",
    "year",
    "observation_count",
    "observation_percent",
    "valid_day_count",
    "arithmetic_mean",
    "arithmetic_standard_dev",
    "first_max_value",
    "ninety_five_percentile",
    "fifty_percentile",
]

CATEGORY_COLUMNS = [
    "parameter_name",
    "units_of_measure",
    "sample_duration",
    "state_name",
]

# Extra columns we want to inspect but may not use as features
EXTRA_COLUMNS = ["state_code", "county_code", "site_number", "datum"]

USECOLS = sorted(set(NUMERIC_COLUMNS + CATEGORY_COLUMNS + EXTRA_COLUMNS))

# Columns that are likely leakage if predicting arithmetic_mean
POTENTIAL_LEAKAGE_COLUMNS = [
    "arithmetic_standard_dev",
    "first_max_value",
    "ninety_five_percentile",
    "fifty_percentile",
    "observation_count",
    "valid_day_count",
    "observation_percent",
]


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def _ensure_output_dir() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def _update_counter(counter: Counter, values: pd.Series) -> None:
    counter.update(values.dropna().astype(str).str.strip())


def _save_fig(path: Path) -> None:
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.close()


# ---------------------------------------------------------------------------
# Step 1: Read data in chunks and collect metadata
# ---------------------------------------------------------------------------

def _read_chunks() -> tuple[dict, pd.DataFrame]:
    """
    Read the raw CSV in chunks, collecting:
    - total row count
    - missing-value counts per column
    - duplicate row count
    - category frequency counters
    - numeric arrays (for full-data statistics)
    - a stratified sample per pollutant for visualisation
    """
    if not RAW_PATH.exists():
        raise FileNotFoundError(f"Raw EPA dataset not found: {RAW_PATH}")

    row_count = 0
    duplicate_count = 0
    missing = Counter()
    coercion_counts = Counter()
    category_counts = {col: Counter() for col in CATEGORY_COLUMNS}
    numeric_chunks: dict[str, list[pd.Series]] = {col: [] for col in NUMERIC_COLUMNS}
    sample_parts: list[pd.DataFrame] = []

    for chunk in pd.read_csv(
        RAW_PATH, usecols=USECOLS, chunksize=CHUNK_SIZE, low_memory=False
    ):
